# Autograd: PyTorch's automatic differentiation

_PyTorch_

**PyTorch records every operation, then computes all the gradients for you with one .backward() call.**

When you do math on a tensor that has requires_grad=True, PyTorch quietly records each operation into a computational graph &mdash; a directed graph of who-was-computed-from-whom. The graph is dynamic: it is built fresh, on the fly, as your Python code runs. Loops, if branches, varying shapes &mdash; all fine, because the graph is whatever your code actually did this time.

       Calling .backward() on a scalar output walks that graph backwards, applying the chain rule at each node, and deposits the gradient into every leaf's .grad. That backward walk is exactly reverse-mode automatic differentiation &mdash; which is exactly backpropagation (see dl-backprop).

---

This notebook is a practice scaffold for the **Autograd: PyTorch's automatic differentiation** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch

# --- 1. A leaf tensor we want gradients for ---
x = torch.tensor(2.0, requires_grad=True)

# --- 2. Compute something. PyTorch records a dynamic graph as we go. ---
y = x**3 + 2*x                      # y = 8 + 4 = 12
print("y          =", y.item())     # 12.0
print("y.grad_fn  =", y.grad_fn)    # <AddBackward0> : the graph edge that built y

# --- 3. Reverse-mode autodiff (backprop): fills x.grad ---
y.backward()
print("x.grad     =", x.grad.item())          # 14.0
print("analytic   =", (3*x.item()**2 + 2))    # 3*4 + 2 = 14.0  -> they match

# --- 4. GRADIENTS ACCUMULATE: backward ADDS into .grad ---
y2 = x**3 + 2*x
y2.backward()
print("after 2nd backward (no zero):", x.grad.item())  # 28.0  (14 + 14) -- the #1 bug!

# --- 5. The fix: zero the gradient before the next backward ---
x.grad.zero_()                      # in a model you'd call optimizer.zero_grad()
y3 = x**3 + 2*x
y3.backward()
print("after zero_grad + backward :", x.grad.item())   # 14.0  -- correct again

# --- 6. torch.no_grad(): stop tracking (inference / freezing) ---
with torch.no_grad():
    z = x**3 + 2*x                  # computed, but NO graph is built
print("z.requires_grad =", z.requires_grad)  # False -- saves memory at inference
print("z.grad_fn       =", z.grad_fn)         # None

# (.detach() does the same for a single tensor: x.detach() shares data but is graph-free)


## Visualize the data & results

_Does PyTorch's autograd actually compute the true derivative? Plot the autograd gradient of f(x)=x^3+2x against the analytic derivative 3x^2+2._

In [ ]:
import numpy as np
import torch

xs = np.linspace(-3, 3, 31)

# Analytic derivative of f(x) = x^3 + 2x  ->  3x^2 + 2
analytic = 3 * xs**2 + 2

# torch.autograd: differentiate the SAME function at each x and read x.grad
autograd = []
for xv in xs:
    x = torch.tensor(float(xv), requires_grad=True)
    y = x**3 + 2*x
    y.backward()
    autograd.append(x.grad.item())
autograd = np.array(autograd)

print("max abs difference:", np.max(np.abs(analytic - autograd)))  # ~0.0
for xv, a, g in zip(xs[::5], analytic[::5], autograd[::5]):
    print(f"x={xv:+.1f}  analytic={a:6.2f}  autograd={g:6.2f}")


## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Type this in Colab. Hand-check autograd. Create x = torch.tensor(2.0, requires_grad=True), compute y = x**3 + 2*x, call y.backward(), and print x.grad. Predict the value first using the derivative $3x^2+2$ at $x=2$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Set requires_grad=True on the leaf, then compute y. — _Only tracked tensors build a graph that .backward() can walk._
- Call y.backward() and read x.grad. — _Reverse-mode autodiff deposits $dy/dx = 3x^2+2 = 14$ into the leaf's .grad._

**Answer:** x = torch.tensor(2.0, requires_grad=True)
y = x**3 + 2*x
y.backward()
print(x.grad)        # tensor(14.)   -- matches 3*2^2 + 2 = 14

</details>

**Problem 2.** Type this in Colab. Inspect the graph. Create x = torch.tensor(3.0, requires_grad=True), compute y = x**2 + 5*x, and print y.grad_fn. Then call y.backward() and print x.grad. Predict x.grad from $2x+5$ at $x=3$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Print y.grad_fn before backward. — _A tracked op attaches a grad_fn (here AddBackward0) — the edge of the graph._
- Call y.backward() and read x.grad. — _Autograd computes $2x+5 = 11$ at $x=3$._

**Answer:** x = torch.tensor(3.0, requires_grad=True)
y = x**2 + 5*x
print(y.grad_fn)     # <AddBackward0 object at 0x...>
y.backward()
print(x.grad)        # tensor(11.)   -- 2*3 + 5 = 11

</details>

**Problem 3.** Type this in Colab. The #1 pitfall: gradients accumulate. With x = torch.tensor(2.0, requires_grad=True), run y = x**3 + 2*x; y.backward() THREE times in a loop WITHOUT zeroing, printing x.grad each time. Predict the three values before running.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Re-run forward + backward() three times without zeroing. — _.backward() ADDS into .grad rather than replacing it._
- Watch x.grad grow 14, 28, 42. — _Each pass deposits the same gradient 14, so they pile up — this is the bug behind broken training loops._

**Answer:** x = torch.tensor(2.0, requires_grad=True)
for i in range(3):
    y = x**3 + 2*x
    y.backward()
    print(x.grad)
# tensor(14.)
# tensor(28.)   -- accumulated!
# tensor(42.)

</details>

**Problem 4.** Type this in Colab. Now fix the accumulation. Repeat the previous loop but call x.grad.zero_() at the top of each iteration (after the first, since .grad starts as None). Print x.grad each time and confirm it stays 14.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Zero the gradient before each backward() (guard the first None). — _Zeroing clears stale gradients so each step sees only its own — in a model this is optimizer.zero_grad()._
- Confirm x.grad is 14 every iteration. — _With a clean slate each pass, the gradient no longer piles up._

**Answer:** x = torch.tensor(2.0, requires_grad=True)
for i in range(3):
    if x.grad is not None:
        x.grad.zero_()          # optimizer.zero_grad() in a real loop
    y = x**3 + 2*x
    y.backward()
    print(x.grad)
# tensor(14.)
# tensor(14.)   -- correct every time
# tensor(14.)

</details>

**Problem 5.** Type this in Colab. Calling .backward() twice on one graph. Create x = torch.tensor(2.0, requires_grad=True), compute y = x**3 + 2*x, call y.backward(), then call y.backward() AGAIN inside a try/except and print the error. Then show retain_graph=True on the first call avoids it.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Call y.backward() twice and catch the RuntimeError. — _PyTorch frees the graph after the first backward to save memory, so a second call fails._
- Pass retain_graph=True on the first backward. — _It keeps the graph alive so a genuine second backward can run._

**Answer:** x = torch.tensor(2.0, requires_grad=True)
y = x**3 + 2*x
y.backward()
try:
    y.backward()                # graph already freed
except RuntimeError as err:
    print("backward twice failed:", "backward through the graph a second time" in str(err))
    # backward twice failed: True

x = torch.tensor(2.0, requires_grad=True)
y = x**3 + 2*x
y.backward(retain_graph=True)   # keep the graph
y.backward()                    # now this works
print(x.grad)                   # tensor(28.)  -- 14 + 14 (still accumulates)

</details>

**Problem 6.** Type this in Colab. Inference with torch.no_grad(). Create x = torch.tensor(2.0, requires_grad=True). Inside with torch.no_grad(): compute z = x**3 + 2*x. Print z.requires_grad and z.grad_fn. Predict both before running.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compute inside with torch.no_grad():. — _It turns off graph construction, so the output is detached._
- Print z.requires_grad and z.grad_fn. — _No graph means requires_grad=False and grad_fn=None — saving memory at inference._

**Answer:** x = torch.tensor(2.0, requires_grad=True)
with torch.no_grad():
    z = x**3 + 2*x
print(z.requires_grad)   # False
print(z.grad_fn)         # None

</details>

**Problem 7.** Type this in Colab. The float-only rule. Try w = torch.tensor([1, 2, 3], requires_grad=True) in a try/except and print the error. Then fix it with float values and confirm w.requires_grad and w.dtype.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Attempt an int tensor with requires_grad=True and catch the error. — _Only floating-point tensors can carry gradients, because gradients are real-valued._
- Use [1.0, 2.0, 3.0] (or dtype=torch.float32). — _Floats can track grad; print requires_grad and dtype to confirm._

**Answer:** try:
    w = torch.tensor([1, 2, 3], requires_grad=True)
except RuntimeError as err:
    print("int failed:", "floating point" in str(err))  # int failed: True
w = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
print(w.requires_grad)   # True
print(w.dtype)           # torch.float32

</details>

**Problem 8.** Type this in Colab. A one-step gradient descent by hand using autograd. Start w = torch.tensor(5.0, requires_grad=True), define loss L = (w - 3)**2, call L.backward(), then update w by w.data -= 0.1 * w.grad inside torch.no_grad(). Print the gradient and the new w.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Backprop the loss to get w.grad. — _$dL/dw = 2(w-3) = 4$ at $w=5$ — the slope pointing uphill._
- Step downhill: w.data -= 0.1 * w.grad. — _Subtracting a fraction of the gradient moves w toward the minimum at 3; this is one optimizer step done by hand._

**Answer:** w = torch.tensor(5.0, requires_grad=True)
L = (w - 3)**2
L.backward()
print(w.grad)            # tensor(4.)   -- 2*(5-3)
with torch.no_grad():
    w -= 0.1 * w.grad     # gradient-descent step
print(w)                 # tensor(4.6000, requires_grad=True)  -- moved toward 3

</details>